# PSLE Math Study Bot — Pipeline Demo & Evaluation Report**AAI3008 Large Language Models Projects — Group 7**| Name | Student ID | Role ||------|-----------|------|| Cheong Wai Hong Jared | 2401641 | Ingestion + chunking + metadata || Chan Jing Chun | 2402867 | Embeddings + indexing + retrieval || Ng Kee Tian | 2401683 | Prompting + grounded answers + citations || Wong Liang Jin | 2400598 | Practice mode + extension features || Lee Xu Xiang, Keith | 2400845 | UI + integration + evaluation harness |---

## Table of Contents1. [Project Overview](#1-project-overview)2. [Setup & Configuration](#2-setup--configuration)3. [Pipeline Walkthrough](#3-pipeline-walkthrough)   - 3.1 Ingestion & Chunking   - 3.2 Topic Classification   - 3.3 Retrieval (FAISS + Reranking)   - 3.4 Generation (RAG + Agent)4. [Evaluation Results](#4-evaluation-results)   - 4.1 Single-Provider Evaluation   - 4.2 Multi-Provider LLM Comparison5. [Feature Showcase](#5-feature-showcase)   - 5.1 Practice Question Generation   - 5.2 Auto-Marking   - 5.3 Progressive Hints6. [Reflection](#6-reflection)

---## 1. Project OverviewWe built a **PSLE Math Study Bot** that helps Primary 5–6 students in Singapore learn math through:- **RAG-powered Q&A** — Retrieves similar worked examples from 7,473 GSM8K problems, then generates step-by-step solutions grounded in those examples.- **Topic-filtered retrieval** — Questions are classified into 6 PSLE math topic families (Percentage, Fractions & Decimals, Ratio & Proportion, Rate, Measurement, Data Handling).- **Lexical reranking + diversity selection** — Retrieved candidates are reranked using semantic + keyword overlap, then selected for parent-document diversity.- **ReAct tool agent** — A safe AST-based calculator tool is invoked for arithmetic-heavy questions.- **Practice mode** — Random GSM8K questions + AI-generated questions with auto-marking, hints, MCQ, and mistake explanations.- **Multi-provider LLM support** — Tested with Gemini, OpenAI, Groq (Llama), and Ollama (local Llama).### Architecture```Student Question → Topic Classifier → Embeddings → FAISS Search    → Lexical Reranking → Diversity Selection → Confidence Check    → [Agent Router] → RAG Path or ReAct Agent Path    → Answer + Citations```

---## 2. Setup & Configuration

In [ ]:
import sysimport osimport jsonimport warningswarnings.filterwarnings("ignore")# Make sure we can import from src/sys.path.insert(0, os.path.abspath(".."))  # If notebook is in notebooks/sys.path.insert(0, os.path.abspath("."))   # If notebook is in repo rootprint("Python version:", sys.version.split()[0])print("Working directory:", os.getcwd())

In [ ]:
# Verify all modules import correctlyfrom src.topic_classifier import classify_question, get_topic_display_name, get_all_topicsfrom src.retrieval import retrieve_with_scores, retrieve_by_topic, load_indexfrom src.ingest import load_gsm8k_docsfrom src.practice import get_random_question, get_final_answer, estimate_difficultyfrom src.tools import calculator, is_calculation_heavyfrom src.generation import get_current_providerprint("All modules imported successfully!")print(f"Current LLM provider: {get_current_provider()}")

---## 3. Pipeline WalkthroughLet's trace a single question through the entire RAG pipeline, step by step.

### 3.1 Ingestion & ChunkingThe GSM8K dataset is loaded, cleaned (removing `<<calc>>` annotations), classified by topic, and stored as LangChain Documents. We support 3 chunking modes: `full`, `step` (sliding window over solution steps), and `hybrid`.

In [ ]:
# Show how a raw GSM8K example gets processedfrom datasets import load_datasetdataset = load_dataset("openai/gsm8k", "main", split="train")raw_example = dataset[0]print("=== RAW GSM8K EXAMPLE ===")print(f"Question: {raw_example['question']}")print(f"\nAnswer:\n{raw_example['answer']}")

In [ ]:
# Show what it looks like after our ingestion pipelinefrom src.ingest import _clean_text, _extract_solution_partscleaned_answer = _clean_text(raw_example['answer'])step_lines, solution_text, final_answer = _extract_solution_parts(raw_example['answer'])print("=== AFTER CLEANING ===")print(f"Solution steps ({len(step_lines)} steps):")for i, step in enumerate(step_lines, 1):    print(f"  Step {i}: {step}")print(f"\nFinal answer: {final_answer}")# Show topic classificationtopic = classify_question(raw_example['question'])print(f"\nClassified topic: {topic} ({get_topic_display_name(topic)})")

### 3.2 Topic ClassificationOur keyword + regex classifier maps questions to 6 PSLE topic families. Patterns are weighted higher (3 points) than keywords (1 point) to reduce false positives.

In [ ]:
# Test classification on sample questions from each topictest_questions = [    ("Find 25% of 80.", "percentage"),    ("Calculate 1/4 + 2/4.", "fractions_decimals"),    ("Divide $84 between Ali and Ben in the ratio 3:4.", "ratio_proportion"),    ("A car travels 180 km in 3 hours. What is its average speed?", "rate"),    ("A rectangle has length 8 cm and width 5 cm. Find its area.", "measurement"),    ("Find the mean of 6, 8, 10, and 12.", "data_handling"),]print(f"{'Question':<60} {'Expected':<20} {'Got':<20} {'OK?'}")print("-" * 105)correct = 0for q, expected in test_questions:    got = classify_question(q)    ok = "PASS" if got == expected else "FAIL"    if got == expected: correct += 1    print(f"{q:<60} {expected:<20} {got:<20} {ok}")print(f"\nAccuracy: {correct}/{len(test_questions)}")

### 3.3 Retrieval (FAISS + Reranking + Diversity Selection)For each query, we:1. Encode with `all-MiniLM-L6-v2` (local, 384-dim embeddings)2. Fetch candidates from FAISS (L2 distance → converted to similarity score)3. Rerank with lexical overlap (20% weight)4. Select diverse results (prefer different parent problems)

In [ ]:
# Demo: retrieve examples for a percentage questionquestion = "A shirt costs $60 and is sold at a 20% discount. How much is the discount?"results = retrieve_with_scores(question, k=4)print(f"Query: {question}")print(f"\nRetrieved {len(results)} examples:\n")for i, (doc, score) in enumerate(results, 1):    topic = doc.metadata.get("topic", "?")    q_preview = doc.metadata.get("question", "")[:80]    chunk_type = doc.metadata.get("chunk_type", "full")    print(f"  {i}. [{get_topic_display_name(topic)}] (sim={score:.3f}, type={chunk_type})")    print(f"     {q_preview}...")    print()

In [ ]:
# Demo: topic-filtered retrievalquestion = "The ratio of boys to girls is 3:2. If there are 15 boys, how many girls?"topic = "ratio_proportion"results = retrieve_by_topic(question, topic, k=4)print(f"Query: {question}")print(f"Topic filter: {get_topic_display_name(topic)}")print(f"\nRetrieved {len(results)} examples:\n")for i, (doc, score) in enumerate(results, 1):    doc_topic = doc.metadata.get("topic", "?")    q_preview = doc.metadata.get("question", "")[:80]    on_topic = "ON-TOPIC" if doc_topic == topic else "OFF-TOPIC"    print(f"  {i}. [{on_topic}] (sim={score:.3f}) {q_preview}...")

### 3.4 Generation (RAG + Agent)The retrieved examples are formatted as context, combined with a PSLE-tutor system prompt, and sent to the LLM. If retrieval confidence is low, we use a fallback prompt without context.For arithmetic-heavy questions, a ReAct agent loop with a safe calculator tool is activated.

In [ ]:
# Demo: safe calculator toolfrom src.tools import calculator, is_calculation_heavy# Test the calculatorexpressions = ["1847 * 293 + 5812", "480 / 8 * 5", "sqrt(144)", "25/100 * 80"]print("=== Calculator Tool ===")for expr in expressions:    result = calculator(expr)    print(f"  {expr} = {result}")# Test the routing heuristicprint("\n=== Agent Routing ===")routing_tests = [    "What is 1847 * 293 + 5812?",    "Find 25% of 80.",    "A car travels 180 km in 3 hours. What is its average speed?",    "Calculate 4523 * 891 - 12345",]for q in routing_tests:    triggers = is_calculation_heavy(q)    route = "AGENT" if triggers else "RAG"    print(f"  [{route}] {q[:60]}")

In [ ]:
# Demo: full end-to-end RAG pipelinefrom src.generation import answer_questionquestion = "A bag originally costs $80. After a 25% discount, what is the sale price?"print(f"Question: {question}\n")result = answer_question(question, topic="percentage")print(f"Provider: {get_current_provider()}")print(f"Confidence: {result['confidence']:.3f}")print(f"Low confidence: {result['low_confidence']}")print(f"Docs retrieved: {result['num_docs_retrieved']}")print(f"Tools used: {result['used_tools']}")print(f"\n{'='*60}")print(f"ANSWER:\n{result['answer'][:500]}")if result['citations']:    print(f"\n{'='*60}")    print(f"CITATIONS ({len(result['citations'])}):")    for i, cite in enumerate(result['citations'], 1):        print(f"  {i}. [{cite['topic']}] {cite['source']} (sim={cite['score']:.0%})")

---## 4. Evaluation Results### 4.1 Single-Provider Evaluation (Quick Mode)Classification and retrieval metrics are **LLM-independent** — they use local embeddings, not the LLM.

In [ ]:
# Run quick evaluation (no LLM calls needed)from src.evaluate import evaluate_topic_classification, evaluate_retrieval_relevance, BENCHMARK_QUESTIONSprint(f"Benchmark size: {len(BENCHMARK_QUESTIONS)} questions")print(f"Topics covered: {len(set(q['topic'] for q in BENCHMARK_QUESTIONS))}")print()# Topic distributionfrom collections import Countertopic_dist = Counter(q['topic'] for q in BENCHMARK_QUESTIONS)for topic, count in topic_dist.most_common():    print(f"  {get_topic_display_name(topic):<25} {count} questions")

In [ ]:
# Eval 1: Topic Classificationclass_results = evaluate_topic_classification(BENCHMARK_QUESTIONS)

In [ ]:
# Eval 2: Retrieval Relevanceret_results = evaluate_retrieval_relevance(BENCHMARK_QUESTIONS)

### 4.2 Multi-Provider LLM ComparisonWe tested our RAG pipeline with 4 different LLM providers to compare answer quality, response time, and cost.> **Key insight:** Classification accuracy (100%) and retrieval precision (67.9%) are **identical** across all providers — these components use local embeddings, not the LLM. The differences only appear in the answer generation step.**To generate comparison data, run these commands in your terminal:**```bashpython -m src.evaluate --provider geminicopy data\benchmark\evaluation_results.json data\benchmark\eval_gemini.jsonpython -m src.evaluate --provider openaicopy data\benchmark\evaluation_results.json data\benchmark\eval_openai.jsonpython -m src.evaluate --provider groqcopy data\benchmark\evaluation_results.json data\benchmark\eval_groq.jsonpython -m src.evaluate --provider ollamacopy data\benchmark\evaluation_results.json data\benchmark\eval_ollama.json```

In [ ]:
# Load and compare results from all providers (run after generating the eval files)import globeval_files = sorted(glob.glob("data/benchmark/eval_*.json"))if not eval_files:    print("No per-provider eval files found yet.")    print("Run the evaluation commands above first, then re-run this cell.")    print()    print("For now, showing the latest evaluation_results.json:")    try:        with open("data/benchmark/evaluation_results.json") as f:            latest = json.load(f)        provider = latest.get("provider", "gemini")        print(f"  Provider: {provider}")        print(f"  Classification: {latest['classification']['accuracy']*100:.1f}%")        if latest.get('retrieval'):            print(f"  Retrieval precision: {latest['retrieval']['topic_precision']*100:.1f}%")            print(f"  Retrieval avg similarity: {latest['retrieval']['avg_similarity']:.3f}")        if latest.get('answer'):            print(f"  Answer correctness: {latest['answer']['accuracy']*100:.1f}%")            print(f"  Avg response time: {latest['answer']['avg_time_seconds']:.1f}s")    except FileNotFoundError:        print("  No evaluation results found. Run: python -m src.evaluate --quick")else:    # Build comparison table    print(f"Found {len(eval_files)} provider evaluations:\n")        header = f"{'Metric':<30}"    rows = {        "Classification accuracy": [],        "Retrieval topic precision": [],        "Retrieval avg similarity": [],        "Answer correctness": [],        "Avg response time (s)": [],    }    providers = []        for path in eval_files:        with open(path) as f:            data = json.load(f)        provider = data.get("provider", path.split("eval_")[1].split(".")[0])        providers.append(provider)        header += f" {provider:>15}"                rows["Classification accuracy"].append(f"{data['classification']['accuracy']*100:.1f}%")        if data.get("retrieval"):            rows["Retrieval topic precision"].append(f"{data['retrieval']['topic_precision']*100:.1f}%")            rows["Retrieval avg similarity"].append(f"{data['retrieval']['avg_similarity']:.3f}")        else:            rows["Retrieval topic precision"].append("N/A")            rows["Retrieval avg similarity"].append("N/A")        if data.get("answer"):            rows["Answer correctness"].append(f"{data['answer']['accuracy']*100:.1f}%")            rows["Avg response time (s)"].append(f"{data['answer']['avg_time_seconds']:.1f}")        else:            rows["Answer correctness"].append("N/A")            rows["Avg response time (s)"].append("N/A")        print(header)    print("-" * len(header))    for metric, values in rows.items():        row = f"{metric:<30}"        for v in values:            row += f" {v:>15}"        print(row)

---## 5. Feature Showcase### 5.1 Practice Question GenerationThe LLM generates fresh practice questions using retrieved examples as style references.

In [ ]:
from src.practice import generate_practice_question# Generate a percentage question at medium difficultyresult = generate_practice_question("percentage", difficulty="medium")print(f"Topic: {result['topic_display']}")print(f"Difficulty: {result['difficulty']}")print(f"\nQuestion: {result['question']}")print(f"\nSolution: {result['solution']}")print(f"\nAnswer: {result['answer']}")

### 5.2 Auto-MarkingThe LLM judges student answers, accepting equivalent forms (e.g., '$12' vs '12', '3/5' vs '0.6').

In [ ]:
from src.generation import auto_mark_answer# Test with a correct answerresult1 = auto_mark_answer(    question="Find 25% of 80.",    correct_answer="20",    student_answer="20")print(f"Correct answer test:")print(f"  Verdict: {'CORRECT' if result1['is_correct'] else 'INCORRECT'}")print(f"  Feedback: {result1['feedback']}")print()# Test with an incorrect answerresult2 = auto_mark_answer(    question="Find 25% of 80.",    correct_answer="20",    student_answer="25")print(f"Incorrect answer test:")print(f"  Verdict: {'CORRECT' if result2['is_correct'] else 'INCORRECT'}")print(f"  Feedback: {result2['feedback']}")

### 5.3 Progressive HintsThree levels of hints, from gentle nudge to partial working.

In [ ]:
from src.generation import generate_hintshints = generate_hints(    question="A shirt costs $60 and is sold at a 20% discount. How much is the discount?",    correct_answer="12")for i, hint in enumerate(hints, 1):    print(f"Hint {i}: {hint}")    print()

---## 6. Reflection### 6.1 How did we use ChatGPT / AI tools in this project?*[Your team should fill this in with specific examples. Below is a template.]*- **Architecture design:** Used ChatGPT to discuss RAG pipeline design choices (chunking strategies, reranking approaches, confidence thresholds).- **Code scaffolding:** Used AI assistance for boilerplate code (Streamlit UI layout, argparse setup, FAISS index management) which we then customised.- **Debugging:** Used ChatGPT to diagnose topic classifier collisions (e.g., the "per" keyword matching inside "percent") and iteratively refined the regex patterns.- **Prompt engineering:** Iterated on the PSLE tutor prompt, auto-marking prompt, and hint generation prompt with AI feedback.- **Evaluation design:** Discussed benchmark question design and answer-matching strategies.**Biggest advantage:** Rapid iteration on prompt templates and classifier rules — what would take hours of trial-and-error took minutes.**Errors discovered:** AI-generated code sometimes had subtle bugs (e.g., the initial `is_calculation_heavy` heuristic triggered on nearly every question, adding unnecessary latency). We caught these through our evaluation suite.### 6.2 If you were to add one new feature, what would it be?**Cross-encoder reranking.** Our current retrieval pipeline uses a bi-encoder (all-MiniLM-L6-v2) for initial retrieval, then a lightweight lexical reranker. A cross-encoder reranker (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`) would score each (query, document) pair jointly, significantly improving retrieval precision — especially for the ratio/proportion questions where our topic precision is weakest (0% for Q21).Our system is **easy to extend** for this: the `retrieve_with_scores()` function already returns scored candidates. We'd add a `_cross_encoder_rerank()` step between `_rerank_results()` and `_select_diverse_results()` — approximately 20 lines of code.### 6.3 As a user of your own system, what feedback would you provide?- **Strengths:** The step-by-step explanations are clear and age-appropriate. The citation system shows students exactly which examples were used. The practice mode with hints and mistake explanations creates a genuine learning loop.- **Weaknesses:** Response time can be slow (3-5s per query) due to the ReAct agent loop adding extra LLM calls. The topic classifier is keyword-based and misses some edge cases. Retrieval precision for ratio/proportion questions is low because the embedding model conflates cooking/recipe problems with ratio problems.- **Suggestions:** Add a "show me a similar problem" button in the Q&A tab (currently only in practice mode). Cache frequently asked questions. Add support for image-based math problems (currently text-only).

---*End of notebook. See the demo video for a live walkthrough of the Streamlit application.*